We have here - 
1) Set up the Neo4j Graph Analytics application within Snowflake.
2) WH, DB, SCHEMA, TABLE & DATA creation into a graph model (users as nodes, transactions as relationships).
3) Ran Node Embeddings and K Nearest Neighbors to identify the structure of nodes in the graph and identify highly similar customer entities.
4) Ran Louvain Community Detection to identify potential clusters of highly similar identities.

In [ ]:
USE ROLE ACCOUNTADMIN;

In [ ]:
--create database
CREATE DATABASE IF NOT EXISTS er_demo;

-- Create an account role to manage the GDS application
CREATE ROLE IF NOT EXISTS gds_role;
GRANT APPLICATION ROLE neo4j_graph_analytics.app_user TO ROLE gds_role;
GRANT APPLICATION ROLE neo4j_graph_analytics.app_admin TO ROLE gds_role;

--Grant permissions for the application to use the database
GRANT USAGE ON DATABASE er_demo TO APPLICATION neo4j_graph_analytics;
GRANT USAGE ON SCHEMA er_demo.public TO APPLICATION neo4j_graph_analytics;

--Create a database role to manage table and view access
CREATE DATABASE ROLE IF NOT EXISTS gds_db_role;

GRANT ALL PRIVILEGES ON FUTURE TABLES IN SCHEMA er_demo.public TO DATABASE ROLE gds_db_role;
GRANT ALL PRIVILEGES ON ALL TABLES IN SCHEMA er_demo.public TO DATABASE ROLE gds_db_role;

GRANT ALL PRIVILEGES ON FUTURE VIEWS IN SCHEMA er_demo.public TO DATABASE ROLE gds_db_role;
GRANT ALL PRIVILEGES ON ALL VIEWS IN SCHEMA er_demo.public TO DATABASE ROLE gds_db_role;

GRANT CREATE TABLE ON SCHEMA er_demo.public TO DATABASE ROLE gds_db_role;


--Grant the DB role to the application and admin user
GRANT DATABASE ROLE gds_db_role TO APPLICATION neo4j_graph_analytics;
GRANT DATABASE ROLE gds_db_role TO ROLE gds_role;

GRANT USAGE ON DATABASE ER_DEMO TO ROLE GDS_ROLE;
GRANT USAGE ON SCHEMA ER_DEMO.PUBLIC TO ROLE GDS_ROLE;

GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA ER_DEMO.PUBLIC TO ROLE GDS_ROLE;
GRANT CREATE TABLE ON SCHEMA ER_DEMO.PUBLIC TO ROLE GDS_ROLE;
GRANT SELECT, INSERT, UPDATE, DELETE ON FUTURE TABLES IN SCHEMA ER_DEMO.PUBLIC TO ROLE GDS_ROLE;

In [ ]:
GRANT ROLE GDS_ROLE TO USER ASHISHCHOPRA;

use warehouse NEO4J_GRAPH_ANALYTICS_APP_WAREHOUSE;
use role gds_role;
use database er_demo;
use schema public;

In [ ]:
CREATE OR REPLACE TABLE node_identities AS
SELECT DISTINCT identity_id AS identity_id FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE node_names AS
SELECT DISTINCT LOWER(first_name || ' ' || last_name) AS full_name FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE node_emails AS
SELECT DISTINCT LOWER(email_address) AS email_id FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE node_phones AS
SELECT DISTINCT phone_number AS phone_number FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE node_birth_years AS
SELECT DISTINCT birth_year AS birth_year FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE node_addresses AS
SELECT DISTINCT LOWER(street_number || ' ' || street_name || ' ' || zip_code) AS full_address FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE node_profiles AS
SELECT DISTINCT profile_id AS profile_id FROM customer_entities_demo_data;

In [ ]:
CREATE OR REPLACE TABLE all_nodes_tbl AS
SELECT identity_id::STRING AS nodeId FROM node_identities
UNION
SELECT full_name::STRING AS nodeId FROM node_names
UNION
SELECT email_id::STRING AS nodeId FROM node_emails
UNION
SELECT phone_number::STRING AS nodeId FROM node_phones
UNION
SELECT birth_year::STRING AS nodeId FROM node_birth_years
UNION
SELECT full_address::STRING AS nodeId FROM node_addresses;

In [ ]:
select * from all_nodes_tbl

In [ ]:
CREATE OR REPLACE TABLE rel_identity_name AS
SELECT 
    identity_id, 
    LOWER(first_name || ' ' || last_name) AS full_name
FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE rel_identity_email AS
SELECT 
    identity_id, 
    LOWER(email_address) AS email_id
FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE rel_identity_phone AS
SELECT 
    identity_id, 
    phone_number
FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE rel_identity_birth_year AS
SELECT 
    identity_id,
    birth_year
FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE rel_identity_address AS
SELECT 
    identity_id,
    LOWER(street_number || ' ' || street_name || ' ' || zip_code) AS full_address
FROM customer_entities_demo_data;

CREATE OR REPLACE TABLE rel_identity_profile AS
SELECT 
    identity_id,
    profile_id
FROM customer_entities_demo_data;

In [ ]:
CREATE OR REPLACE TABLE all_relationships_tbl AS
SELECT identity_id::STRING AS sourceNodeId, full_name::STRING AS targetNodeId FROM rel_identity_name
UNION
SELECT identity_id::STRING, email_id::STRING FROM rel_identity_email
UNION
SELECT identity_id::STRING, phone_number::STRING FROM rel_identity_phone
UNION
SELECT identity_id::STRING, birth_year::STRING FROM rel_identity_birth_year
UNION
SELECT identity_id::STRING, full_address::STRING FROM rel_identity_address;

In [ ]:
select * from all_relationships_tbl

used FastRP - compute embeddings

In [ ]:
CALL Neo4j_Graph_Analytics.graph.fast_rp('CPU_X64_XS', {
  'project': {
    'defaultTablePrefix': 'er_demo.public',
    'nodeTables': ['all_nodes_tbl'],
    'relationshipTables': {
      'all_relationships_tbl': {
        'sourceTable': 'all_nodes_tbl',
        'targetTable': 'all_nodes_tbl',
        'orientation': 'UNDIRECTED'

      }
    }
  },
  'compute': {
    'mutateProperty': 'embedding',
    'embeddingDimension': 128
  },
  'write': [{
    'nodeLabel': 'all_nodes_tbl',
    'outputTable': 'er_demo.public.all_nodes_fast_rp',
    'nodeProperty': 'embedding'
  }]
});

In [ ]:
SELECT
  nodeid,
  embedding
FROM er_demo.public.all_nodes_fast_rp;

used K-Nearest Neighbors (KNN) - helps us find structurally similar identities — even if they’re not directly connected. It compares the cosine similarity of embeddings to rank the top matches for each node.

In [ ]:
CALL Neo4j_Graph_Analytics.graph.knn('CPU_X64_XS', {
  'project': {
    'defaultTablePrefix': 'er_demo.public',
    'nodeTables': [ 'all_nodes_fast_rp' ],
    'relationshipTables': {}
  },
  'compute': {
    'nodeProperties': ['EMBEDDING'],
    'topK': 3,
    'mutateProperty': 'score',
    'mutateRelationshipType': 'SIMILAR_TO'
  },
  'write': [{
    'outputTable': 'er_demo.public.all_nodes_knn_similarity',
    'sourceLabel': 'all_nodes_fast_rp',
    'targetLabel': 'all_nodes_fast_rp',
    'relationshipType': 'SIMILAR_TO',
    'relationshipProperty': 'score'
  }]
});

In [ ]:
SELECT
    TO_DECIMAL(score, 10, 5) AS score_rounded,
    COUNT(*) AS row_count
FROM er_demo.public.all_nodes_knn_similarity
GROUP BY TO_DECIMAL(score, 10, 5)
ORDER BY score_rounded desc;

used Louvain Community Detection - detect communities of likely duplicate records

In [ ]:

CALL Neo4j_Graph_Analytics.graph.louvain('CPU_X64_XS', {
  'project': {
    'defaultTablePrefix': 'er_demo.public',
    'nodeTables': ['all_nodes_tbl'],
    'relationshipTables': {
      'all_nodes_knn_similarity': {
        'sourceTable': 'all_nodes_tbl',
        'targetTable': 'all_nodes_tbl'
      }
    }
  },
  'compute': {
    'mutateProperty': 'community'
  },
  'write': [{
    'outputTable': 'er_demo.public.knn_louvain_communities',
    'nodeLabel': 'all_nodes_tbl',
    'nodeProperty': 'community'
  }]
});

In [ ]:
SELECT
    community as community_id,
    COUNT(*) AS community_size
FROM er_demo.public.knn_louvain_communities
GROUP BY community
ORDER BY community_size desc;